# 5교시 · 데이터 시각화
### — 남에게 보여 주기

앞 시간까지는 **내가 보려고** 숫자를 냈습니다.
이번 시간에는 **남이 보게** 만듭니다. 여기서부터는 성격이 달라집니다.

숫자는 틀리지 않았는데 그림 때문에 **상대가 잘못 이해하는 일**이 자주 생깁니다.
이번 시간에는 그림을 그리는 방법과 함께, **그림이 사람을 오해하게 만드는 방식**도 같이 봅니다.

**이 시간이 끝나면 할 수 있는 것**

1. 말하려는 내용에 맞는 차트 종류를 고를 수 있다
2. Matplotlib 으로 제목·축이름·범례를 직접 붙일 수 있다
3. Seaborn 으로 분포와 관계를 짧은 코드로 그릴 수 있다
4. **같은 데이터로 정반대 인상을 주는 그래프를 만들 수 있고, 그래서 조심할 수 있다**


### 오늘 쓰는 데이터 — 문구·가구 유통사 주문 내역

| | |
|---|---|
| **무엇** | 어느 문구·가구 유통사의 주문 내역 (Tableau 공식 샘플 데이터) |
| **기간** | 2023-01-03 ~ 2026-12-30 (4년치) |
| **크기** | 10,239행 × 21열 · 주문 5,111건 · 고객 804명 |
| **한 행은** | 주문이 아니라 **주문에 담긴 품목 하나**입니다 |
| **지역** | 미국(10,038) · 캐나다(201) |

**주요 열**

| 열 | 뜻 |
|---|---|
| `Order ID` · `Order Date` · `Ship Date` | 주문번호 · 주문일 · 배송일 |
| `Customer ID` · `Segment` | 고객 · 고객 유형(Consumer / Corporate / Home Office) |
| `Region` · `State/Province` · `City` | 지역(Central / East / South / West) · 주 · 도시 |
| `Category` · `Sub-Category` · `Product Name` | 대분류(3종) · 소분류(17종) · 제품명 |
| `Sales` · `Quantity` · `Discount` · `Profit` | 매출 · 수량 · 할인율 · 이익 |

> **결측치·이상치·중복값이 일부러 들어 있습니다.**
> 실무에서 받는 데이터가 그렇기 때문입니다. 손대지 않은 원본이 필요하면
> `superstore_orders_raw.csv` 를 쓰세요.

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders = pd.read_csv(BASE + 'superstore_orders.csv', parse_dates=['Order Date', 'Ship Date'])
orders = orders.drop_duplicates()

print(orders.shape)

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)      # 마이너스 기호도 깨지므로 함께 설정

plt.plot([1, 2, 3], [1, 4, 2])
plt.title('한글이 보이면 성공입니다')
plt.show()

In [ ]:
# 오늘 그림의 재료 — 지역별 매출 합계
by_region = orders.groupby('Region')['Sales'].sum().sort_values(ascending=False)

by_region.round(0)

---
# 5-1. 무엇을 말하려는가에 따라 차트가 정해진다

차트 종류는 수십 가지가 있지만, **업무에서 쓰는 것은 네 가지**입니다.
그리고 그 넷은 **말하려는 내용**으로 구분됩니다.

| 말하려는 것 | 예시 문장 | 차트 | 코드 |
|---|---|---|---|
| **비교** | "동부가 남부보다 두 배 판다" | 막대그래프 | `kind='bar'` |
| **추이** | "매출이 하반기에 올라간다" | 선그래프 | `kind='line'` |
| **관계** | "할인을 많이 하면 이익이 준다" | 산점도 | `kind='scatter'` |
| **분포** | "주문 대부분은 소액이다" | 히스토그램 | `kind='hist'` |

네 가지를 실제로 하나씩 그려 보겠습니다.
항목 이름이 길면 세로 막대(`'bar'`) 대신 **가로 막대(`'barh'`)** 를 쓰고,
그리기 전에 `sort_values()` 로 **정렬**하면 순위가 한눈에 들어옵니다.


## ① 비교 — 막대그래프

**항목끼리 크기를 견주어 볼 때** 씁니다.
막대는 **길이**로 크기를 나타내기 때문에 사람 눈이 가장 정확하게 비교합니다.

In [ ]:
by_subcat = orders.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False).head(10)

by_subcat.plot(kind='barh', figsize=(8, 5))
plt.title('매출 상위 10개 품목')
plt.xlabel('매출')
plt.show()

## ② 추이 — 선그래프

**시간에 따라 어떻게 변했는지** 보여 줄 때 씁니다.
점을 선으로 이어 놓기 때문에 **"이어져 있다"** 는 느낌을 줍니다.
그래서 **가로축이 시간일 때만** 쓰는 것이 원칙입니다.

In [ ]:
orders['year_month'] = orders['Order Date'].dt.to_period('M')
monthly_sales = orders.groupby('year_month')['Sales'].sum()
monthly_sales.index = monthly_sales.index.to_timestamp()      # 그림용으로 날짜형으로 되돌립니다

monthly_sales.plot(kind='line', figsize=(10, 4))
plt.title('월별 매출')
plt.ylabel('매출')
plt.show()

## ③ 관계 — 산점도

**두 숫자가 서로 관련이 있는지** 볼 때 씁니다.
점 하나가 데이터 한 행입니다. 여기서는 점 하나가 **주문 한 건**입니다.

In [ ]:
orders.plot(kind='scatter', x='Discount', y='Profit', figsize=(8, 5), alpha=0.3)
plt.title('할인율과 이익')
plt.axhline(0, color='red', linewidth=1)       # 이익 0 기준선
plt.show()

In [ ]:
print('할인율과 이익의 상관계수: {:.3f}'.format(orders['Discount'].corr(orders['Profit'])))
print('적자 주문 비율: {:.1f}%'.format((orders['Profit'] < 0).mean() * 100))

**상관계수 -0.219.** 마이너스니까 "할인이 커질수록 이익은 작아지는 쪽"입니다.
다만 -1 에서 한참 먼 값이라 **약한 관계**입니다. 할인만으로 이익이 정해지지는 않습니다.

> 상관계수는 -1 ~ +1 사이 값입니다. 0 에 가까우면 관계가 거의 없고,
> ±1 에 가까우면 한쪽이 변할 때 다른 쪽도 규칙적으로 변합니다.
>
> **그리고 관계가 있다는 것이 원인이라는 뜻은 아닙니다.**
> 할인을 많이 해서 이익이 준 것인지, 원래 안 팔리는 물건이라 할인을 많이 한 것인지
> 이 숫자만으로는 알 수 없습니다. **상관관계는 인과관계가 아닙니다.**


In [ ]:
# ④ 분포 — 히스토그램
orders[orders['Sales'] < 1000]['Sales'].plot(kind='hist', bins=50, figsize=(9, 4))
plt.title('주문 금액 분포 (1,000 미만)')
plt.xlabel('주문 금액')
plt.show()

---
# 5-2. Matplotlib 기본 — 직접 그리기

지금까지는 `df.plot(...)` 을 썼습니다. **판다스가 대신 그려 준 것**입니다.
편하지만, 제목·축 이름·범례를 붙이려면 **Matplotlib 을 직접** 쓰는 편이 낫습니다.

`df.plot()` 도 사실은 내부에서 Matplotlib 을 부릅니다.
**같은 도구인데, 판다스를 거치느냐 직접 부르느냐의 차이**입니다.


In [ ]:
plt.figure(figsize=(8, 5))                        # ① 그림판 크기를 먼저 정합니다

plt.bar(by_region.index, by_region.values, color='#4C72B0')   # ② 막대를 그립니다

plt.title('동부 지역이 전체 매출의 32.9%를 차지합니다')   # ③ 제목
plt.xlabel('지역')                                 # ④ 가로축 이름
plt.ylabel('매출')                                 # ⑤ 세로축 이름

plt.show()                                        # ⑥ 화면에 띄웁니다

## 한 줄씩 무슨 뜻인지

| 코드 | 하는 일 |
|---|---|
| `plt.figure(figsize=(8, 5))` | **그림판을 준비합니다.** 가로 8, 세로 5 (인치 단위) |
| `plt.bar(x, y)` | 막대를 그립니다. 선은 `plt.plot`, 점은 `plt.scatter` |
| `color='#4C72B0'` | 색을 지정합니다. `'red'` 처럼 이름으로도, `'#4C72B0'` 처럼 코드로도 됩니다 |
| `plt.title('...')` | 제목을 붙입니다 |
| `plt.xlabel` / `plt.ylabel` | 가로축·세로축 이름을 붙입니다 |
| `plt.show()` | **여기까지 그린 것을 화면에 띄웁니다.** 이걸 부르면 그림판이 비워집니다 |

**`plt.show()` 를 부르기 전까지 명령이 계속 같은 그림에 쌓입니다.**
그래서 `plt.bar` 로 막대를 그리고, 그 뒤에 `plt.title` 로 제목을 얹는 게 가능합니다.

`plt.show()` 를 안 쓰면 다음 셀의 그림과 겹쳐 그려질 수 있습니다. **항상 마지막에 넣으세요.**

## 범례 — 선이 여러 개일 때

한 그림에 선을 여러 개 그리면 **어느 선이 뭔지** 알려 줘야 합니다.
그게 **범례(legend)** 입니다.

`label=` 로 이름을 붙이고 `plt.legend()` 를 부르면 됩니다.

In [ ]:
cat_monthly = orders.pivot_table(index='year_month', columns='Category',
                                values='Sales', aggfunc='sum')
cat_monthly.index = cat_monthly.index.to_timestamp()

plt.figure(figsize=(11, 4))

for name in ['Furniture', 'Office Supplies', 'Technology']:
    plt.plot(cat_monthly.index, cat_monthly[name], label=name)

plt.title('카테고리별 월 매출 추이')
plt.ylabel('매출')
plt.legend()                    # 이 한 줄이 범례를 만듭니다
plt.grid(alpha=0.3)             # 옅은 격자선 (값을 읽기 쉬워집니다)
plt.show()

---
# 5-3. Seaborn — 짧은 코드로 분포와 관계

**Seaborn(시본)** 은 Matplotlib 위에 얹혀 있는 도구입니다.
Matplotlib 으로 열 줄 걸리는 것을 한 줄로 그려 줍니다. **Colab 에는 이미 설치돼 있습니다.**

Seaborn 함수들은 대부분 이 모양입니다.

```python
sns.무슨그림(data=표, x='가로축열', y='세로축열')
```

**표를 통째로 넘기고, 열 이름만 알려 주면 됩니다.**


In [ ]:
import seaborn as sns

sns.set_theme(style='whitegrid', font='NanumGothic')   # 보기 좋은 기본 설정 + 한글 폰트
plt.rc('axes', unicode_minus=False)

print(sns.__version__)

In [ ]:
# ① 분포 — `sns.histplot`
plt.figure(figsize=(9, 4))

sns.histplot(data=orders[orders['Sales'] < 1000], x='Sales', bins=50)

plt.title('주문의 절반이 53.7 이하입니다')
plt.xlabel('주문 금액')
plt.ylabel('주문 건수')
plt.show()

In [ ]:
# ③ 관계 — `sns.scatterplot`
plt.figure(figsize=(9, 5))

sns.scatterplot(data=orders, x='Discount', y='Profit',
                hue='Category', alpha=0.4)

plt.title('할인율이 높아질수록 적자 주문이 늘어납니다')
plt.axhline(0, color='red', linewidth=1)
plt.show()

In [ ]:
# ④ 비교 — sns.barplot (기본은 평균이므로 estimator="sum" 으로 합계 지정)
plt.figure(figsize=(8, 4))

sns.barplot(data=orders, x='Region', y='Sales',
            estimator='sum', errorbar=None,
            order=['East', 'West', 'Central', 'South'])   # 큰 순서로 직접 지정

plt.title('지역별 매출 합계')
plt.ylabel('매출 합계')
plt.show()

In [ ]:
# ⑤ 상자그림 — `sns.boxplot`
plt.figure(figsize=(9, 5))

sns.boxplot(data=orders[orders['Sales'] < 1000], x='Category', y='Sales')

plt.title('카테고리별 주문 금액 (1,000 미만)')
plt.ylabel('주문 금액')
plt.show()

## Matplotlib 과 Seaborn, 언제 뭘 쓰나

| | Matplotlib | Seaborn |
|---|---|---|
| 코드 길이 | 길다 | 짧다 |
| 그룹별로 나눠 그리기 | 반복문 필요 | `hue=` 한 줄 |
| 세밀하게 손보기 | 자유롭다 | 제한이 있다 |
| 기본 모양 | 밋밋하다 | 보기 좋다 |

**둘 중 하나를 고르는 게 아닙니다.**
Seaborn 으로 그려 놓고 `plt.title` · `plt.ylabel` 로 다듬는 것이 실제로 가장 흔한 방식입니다.
위 코드들도 전부 그렇게 했습니다.

---
# 5-4. 실습 — 오늘 배운 다섯 가지

아래 다섯 문제의 **빈칸(`____`)을 채우고 실행**하세요.
각 문제는 오늘 배운 차트 하나씩에 대응합니다.


In [ ]:
# 문제 1. 막대 — 고객 유형(Segment)별 매출 합계를 막대그래프로 그리세요.
seg = orders.groupby('Segment')['Sales'].sum()

seg.plot(kind='bar', figsize=(7, 3))
plt.title('고객 유형별 매출')
plt.ylabel('매출')
plt.show()

In [ ]:
# 문제 2. 가로 막대 — 같은 자료를 큰 순서로 정렬해 가로 막대로 그리세요.
seg.sort_values().plot(kind='barh', figsize=(7, 3))
plt.title('고객 유형별 매출')
plt.xlabel('매출')
plt.show()

In [ ]:
# 문제 3. 선그래프 — 월별 매출을 선그래프로 그리세요.
m = orders.groupby('year_month')['Sales'].sum()
m.index = m.index.to_timestamp()

m.plot(kind='line', figsize=(10, 3))
plt.title('월별 매출')
plt.ylabel('매출')
plt.show()

In [ ]:
# 문제 4. 산점도 — 할인율(Discount)과 이익(Profit)의 관계를 산점도로 그리세요.
orders.plot(kind='scatter', x='Discount', y='Profit', alpha=0.2, figsize=(8, 4))
plt.title('할인율과 이익')
plt.axhline(0, color='red', linewidth=1)
plt.show()

In [ ]:
# 문제 5. 축 범위 — 막대그래프의 세로축을 0부터 그리세요.
seg.plot(kind='bar', figsize=(7, 3))
plt.ylim(0, seg.max() * 1.1)
plt.title('세로축을 0부터 그린 정직한 막대')
plt.ylabel('매출')
plt.show()

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 판다스로 빠르게 그리기 | `df['열'].plot(kind='bar')` |
| 그림판 크기 정하기 | `plt.figure(figsize=(8, 5))` |
| 막대 / 선 / 점 | `plt.bar(x, y)` · `plt.plot(x, y)` · `plt.scatter(x, y)` |
| 제목 · 축 이름 | `plt.title(...)` · `plt.xlabel(...)` · `plt.ylabel(...)` |
| 범례 | `plt.plot(..., label='이름')` + `plt.legend()` |
| 축 범위 | `plt.ylim(아래, 위)` |
| 기준선 | `plt.axhline(0)` · `plt.axvline(0)` |
| 화면에 띄우기 | `plt.show()` |
| Seaborn 기본 설정 | `sns.set_theme(style='whitegrid', font='NanumGothic')` |
| 분포 | `sns.histplot(data=df, x='열', hue='그룹')` |
| 관계 | `sns.scatterplot(data=df, x='열1', y='열2', hue='그룹')` |
| 비교 | `sns.barplot(data=df, x='그룹', y='값', estimator='sum')` |
| 상자그림 | `sns.boxplot(data=df, x='그룹', y='값')` |
| 한글 폰트 | `!apt-get -qq install fonts-nanum` + `plt.rc('font', family='NanumGothic')` |

## 차트 고르는 표

| 말하려는 것 | 차트 |
|---|---|
| 항목끼리 비교 | 막대 (항목 이름이 길면 가로 막대) |
| 시간에 따른 변화 | 선 |
| 두 숫자의 관계 | 산점도 |
| 값이 퍼진 모양 | 히스토그램 · 상자그림 |

## 남길 것 세 가지

1. **차트 종류는 말하려는 내용이 정한다** — 비교는 막대, 추이는 선, 관계는 산점도, 분포는 히스토그램
2. **제목과 축 이름을 반드시 붙인다** — 단위 없는 그림은 읽는 사람이 해석할 수 없습니다
3. **막대그래프의 세로축은 0에서 시작한다** — 막대는 길이로 크기를 말하므로, 잘라내면 비율이 실제와 달라집니다

---

### 다음 시간

**남은 시간은 오늘 배운 것을 실제 분석에 써 보는 데 씁니다.**

지금까지 배운 조회·집계·탐색·시각화를 하나의 질문에 이어 붙여,
**처음부터 끝까지 하나의 분석**을 해 봅니다.
